# EasyPiper Functional Test

A quick, safe test suite for the `piper_sdk.easy_piper.EasyPiper` wrapper.

- Arm: enable/disable, emergency stop/resume, reset sequence
- Modes: Joint (MOVE J), Point-to-Point (MOVE P), Linear (MOVE L)
- Motions: go to joint zero, go to TCP zero, go to specific joints/TCP pose
- Gripper: enable, set zero, disable

Units:
- Joints: degrees (converted to 0.001° for SDK)
- TCP: mm/deg (converted to 0.001 mm / 0.001° for SDK)

In [ ]:
import time
from easy_piper import EasyPiper

# Create and auto-connect (calls ConnectPort and waits briefly)
ep = EasyPiper(auto_connect=True)

# Optional: show firmware version if available
try:
    print('Firmware:', ep.iface.GetPiperFirmwareVersion())
except Exception as e:
    print('Firmware read skipped:', e)

## 1) Safe reset and enable
Sequence: emergency stop → resume → enable → switch to MOVE J at moderate speed.

In [ ]:
ep.reset_sequence(speed_percent=30)
m = ep.get_current_mode()
print('Mode after reset:', m.mode_ctrl.ctrl_mode, m.mode_ctrl.move_mode, m.mode_ctrl.move_spd_rate_ctrl, 'Hz:', m.Hz)

## 2) Go to joint zero
All joints → 0°.

In [ ]:
ep.go_zero_joints(ensure_joint_mode=True, speed_percent=30)
time.sleep(1.0)
print('Moved to joint zero.')

## 3) Go to TCP zero (MOVE L)
Moves to the default TCP pose corresponding to joint zero.

In [ ]:
ep.go_zero_tcp(mode='L', speed_percent=30)
time.sleep(1.0)
print('TCP zero (MOVE L). Current TCP:', ep.get_current_tcp())

## 4) Mode switching checks
Switch Joint → MOVE P → MOVE L and print current mode each time.

In [ ]:
ep.switch_mode_joint(30); time.sleep(0.1); m = ep.get_current_mode(); print('Joint mode:', m.mode_ctrl.ctrl_mode, m.mode_ctrl.move_mode, m.mode_ctrl.move_spd_rate_ctrl)
ep.switch_mode_move_p(30); time.sleep(0.1); m = ep.get_current_mode(); print('MOVE P   :', m.mode_ctrl.ctrl_mode, m.mode_ctrl.move_mode, m.mode_ctrl.move_spd_rate_ctrl)
ep.switch_mode_move_l(30); time.sleep(0.1); m = ep.get_current_mode(); print('MOVE L   :', m.mode_ctrl.ctrl_mode, m.mode_ctrl.move_mode, m.mode_ctrl.move_spd_rate_ctrl)

## 5) Joint move (small) then return to zero

In [ ]:
ep.switch_mode_joint(30)
ep.go_to_joint_angles([5.0, -5.0, 0.0, 0.0, 0.0, 0.0])
time.sleep(1.0)
ep.go_zero_joints()
time.sleep(1.0)
print('Joint move done, returned to zero.')

## 6) TCP move in MOVE P (small offset)
Move a gentle +20mm X and +20mm Z relative to the default TCP zero pose, then return.

In [ ]:
# Move to TCP zero in MOVE P first
ep.go_zero_tcp(mode='P', speed_percent=30)
time.sleep(1.0)

# Default pose used by EasyPiper when joints are zero
X0, Y0, Z0, RX0, RY0, RZ0 = ep.tcp_zero_pose

# Move to small offset, then back
ep.go_to_tcp_pose(X0 + 20.0, Y0 + 0.0, Z0 + 20.0, RX0, RY0, RZ0, ensure_mode=None, speed_percent=30, wait_s=1.0)
ep.go_to_tcp_pose(X0 + 0.0,  Y0 + 0.0, Z0 + 0.0,  RX0, RY0, RZ0, ensure_mode=None, speed_percent=30, wait_s=1.0)
print('TCP MOVE P small offset done.')

## 7) Emergency stop and resume
Demonstrate E-stop → resume → ensure enabled.

In [ ]:
ep.emergency_stop(); time.sleep(0.05)
# ep.resume(); time.sleep(0.05)
# print('Resumed. Enabling motors...')
# print('Enabled:', ep.enable(wait=True))

## 8) Gripper: enable → set zero → disable

In [ ]:
ep.gripper_enable(effort=800, clear_error=True); time.sleep(0.2)
ep.gripper_set_zero(); time.sleep(0.2)
ep.gripper_disable(clear_error=True)
print('Gripper sequence complete.')

## 9) Introspection
Read and display current mode, TCP, and (raw) joint state.

In [ ]:
m = ep.get_current_mode()
tcp = ep.get_current_tcp()
j = ep.get_joint_state()
print('Mode:', m.mode_ctrl.ctrl_mode, m.mode_ctrl.move_mode, m.mode_ctrl.mit_mode, m.mode_ctrl.move_spd_rate_ctrl, 'Hz:', m.Hz)
print('TCP :', tcp)
print('Joint state type:', type(j))
print('Joint state dir :', [a for a in dir(j) if not a.startswith('_')])

## 10) Optional: Disable motors

In [ ]:
print('Disabling motors...')
ep.disable()
print('Done.')

In [ ]:
print(ep.get_current_mode())
print(ep.get_current_tcp())

In [ ]:
ep.set_slave_arm()
print('Set to slave arm mode.') 

In [ ]:
print(ep.is_ok)